In [ ]:
import pandas as pd
import numpy as np

# ==========================================
# 1. Data handling and preprocessing
# ==========================================
df = pd.read_csv('cleaned_data.csv')
df['Date'] = pd.to_datetime(df['Date'])
# Must sort by date, otherwise feature calculations will be incorrect
df = df.sort_values('Date').reset_index(drop=True)
print("Original Feature. Shape:", df.shape)
df.head(20)

In [ ]:
# ==========================================
# 2. Implied Probability Features
# ==========================================
# Convert odds to implied probabilities (1/Odds)
df['ImpPH'] = 1 / df['AvgH']
df['ImpPD'] = 1 / df['AvgD']
df['ImpPA'] = 1 / df['AvgA']
# Normalize probabilities to remove bookmaker margin
total_prob = df['ImpPH'] + df['ImpPD'] + df['ImpPA']
df['ImpPH'] = df['ImpPH'] / total_prob
df['ImpPD'] = df['ImpPD'] / total_prob
df['ImpPA'] = df['ImpPA'] / total_prob

In [ ]:
# ==========================================
# 3. ELO Rating System
# ==========================================
def calculate_elo_features(df, k_factor=20, initial_elo=1500):
    elo_dict = {}
    home_elos, away_elos = [], []
    elo_probs_h, elo_probs_a, elo_probs_d = [], [], []
    
    for idx, row in df.iterrows():
        h_team, a_team = row['HomeTeam'], row['AwayTeam']
        r_home = elo_dict.get(h_team, initial_elo)
        r_away = elo_dict.get(a_team, initial_elo)
        
        home_elos.append(r_home)
        away_elos.append(r_away)
        
        # ELO(Standard Logistics Curve)
        dr = r_home - r_away + 100 # +100 is a common home advantage compensation
        p_home_win = 1 / (1 + 10**(-dr/400))
        p_away_win = 1 - p_home_win
        
        # Standard ELO does not directly produce draw probabilities.
        # Simulate a draw probability: the closer the teams' ratings, the higher the draw probability.
        p_draw_sim = 1 - abs(p_home_win - 0.5) * 2 # Maximum 1, minimum 0
        
        elo_probs_h.append(p_home_win)
        elo_probs_a.append(p_away_win)
        elo_probs_d.append(p_draw_sim) 
        
        # Post-match update
        if row['FTR'] == 'H': s_h = 1
        elif row['FTR'] == 'D': s_h = 0.5
        else: s_h = 0
        
        # use the expected value without home advantage to update strength.
        e_pure_home = 1 / (1 + 10**(-(r_home - r_away)/400))
        
        elo_dict[h_team] = r_home + k_factor * (s_h - e_pure_home)
        elo_dict[a_team] = r_away + k_factor * ((1-s_h) - (1-e_pure_home))
        
    df['HomeElo'] = home_elos
    df['AwayElo'] = away_elos
    df['EloDiff'] = df['HomeElo'] - df['AwayElo']
    df['EloHomeW'] = elo_probs_h
    df['EloAwayW'] = elo_probs_a
    df['EloDraw'] = elo_probs_d
    return df

df = calculate_elo_features(df)

In [ ]:
# ==========================================
# 4. Long Format
# ==========================================
# Purpose: To calculate historical cumulative data (ODM, win rate, points, etc.)

# Home team perspective
home_df = df[['Date', 'HomeTeam', 'FTHG', 'FTAG', 'FTR', 'HS', 'AS', 'HST', 'AST', 'HC', 'AC', 'HF', 'AF', 'Season']].copy()
home_df.columns = ['Date', 'Team', 'GoalsFor', 'GoalsAgainst', 'Result', 'ShotsFor', 'ShotsAgainst', 'SoTFor', 'SoTAgainst', 'CornersFor', 'CornersAgainst', 'FoulsFor', 'FoulsAgainst', 'Season']
home_df['IsHome'] = 1
home_df['Points'] = home_df['Result'].map({'H':3, 'D':1, 'A':0})
home_df['Result'] = home_df['Result'].map({'H':'W', 'D':'D', 'A':'L'})

# Away team perspective
away_df = df[['Date', 'AwayTeam', 'FTAG', 'FTHG', 'FTR', 'AS', 'HS', 'AST', 'HST', 'AC', 'HC', 'AF', 'HF', 'Season']].copy()
away_df.columns = ['Date', 'Team', 'GoalsFor', 'GoalsAgainst', 'Result', 'ShotsFor', 'ShotsAgainst', 'SoTFor', 'SoTAgainst', 'CornersFor', 'CornersAgainst', 'FoulsFor', 'FoulsAgainst', 'Season']
away_df['IsHome'] = 0
away_df['Points'] = away_df['Result'].map({'A':3, 'D':1, 'H':0})
away_df['Result'] = away_df['Result'].map({'A':'W', 'D':'D', 'H':'L'})

long_df = pd.concat([home_df, away_df], axis=0).sort_values(['Team', 'Date']).reset_index(drop=True)

In [ ]:
# ==========================================
# 4. Features Calculation
# ==========================================

def calculate_all_features(group):
    # Core: Must Shift(1) to prevent future data leakage
    shifted = group.shift(1)
    
    # --- A. Form-based Features (Last 6 Matches) ---
    rolling_6 = shifted[['GoalsFor', 'GoalsAgainst', 'ShotsFor', 'SoTFor', 'CornersFor', 'FoulsFor']].rolling(window=6, min_periods=1).mean()
    rolling_6.columns = ['AvgGoals', 'AvgConceded', 'AvgShots', 'AvgSoT', 'AvgCorners', 'AvgFouls']
    
    # --- B. Statistical Features (All Previous) ---
    is_win = (shifted['Result'] == 'W').astype(float)
    is_draw = (shifted['Result'] == 'D').astype(float)
    
    all_win_rate = is_win.expanding().mean() # HomeW / AwayW
    all_draw_rate = is_draw.expanding().mean() # HomeD / AwayD
    
    # --- C. ODM-based Features (Offensive/Defensive Cap) ---
    off_cap = shifted['GoalsFor'].expanding().mean() # HomeOff
    def_cap = shifted['GoalsAgainst'].expanding().mean() # HomeDef
    
    # --- D. League Point-based Features ---
    curr_points = group.groupby('Season')['Points'].transform(lambda x: x.cumsum().shift(1).fillna(0))
    
    # --- E. Rest Days ---
    rest_days = group['Date'].diff().dt.days
    
    # Combine all features
    feats = pd.concat([rolling_6, all_win_rate, all_draw_rate, off_cap, def_cap, curr_points, rest_days], axis=1)
    feats.columns = list(rolling_6.columns) + ['WinRate', 'DrawRate', 'Off', 'Def', 'P', 'RestDays']
    
    return feats

# Apply calculation
team_stats = long_df.groupby('Team', group_keys=False).apply(calculate_all_features)
long_df = pd.concat([long_df, team_stats], axis=1)

# --- F. Calculate specific home/away win rates (HomeHW, HomeHD, AwayAW, AwayAD) ---
# Need to separately filter home and away games for calculation

# Home team performance at home
home_games = long_df[long_df['IsHome']==1].copy()
# use groupby('Team') to isolate each team's home games, then shift(1) to exclude current match, and expanding().mean() to get cumulative average
home_games['VenueWinRate'] = home_games.groupby('Team')['Result'].transform(lambda x: (x.shift(1) == 'W').expanding().mean())
home_games['VenueDrawRate'] = home_games.groupby('Team')['Result'].transform(lambda x: (x.shift(1) == 'D').expanding().mean())

long_df.loc[home_games.index, 'VenueWinRate'] = home_games['VenueWinRate']
long_df.loc[home_games.index, 'VenueDrawRate'] = home_games['VenueDrawRate']

# Away team performance at away
away_games = long_df[long_df['IsHome']==0].copy()
# use groupby('Team') to isolate each team's away games, then shift(1) to exclude current match, and expanding().mean() to get cumulative average
away_games['VenueWinRate'] = away_games.groupby('Team')['Result'].transform(lambda x: (x.shift(1) == 'W').expanding().mean())
away_games['VenueDrawRate'] = away_games.groupby('Team')['Result'].transform(lambda x: (x.shift(1) == 'D').expanding().mean())

long_df.loc[away_games.index, 'VenueWinRate'] = away_games['VenueWinRate']
long_df.loc[away_games.index, 'VenueDrawRate'] = away_games['VenueDrawRate']

In [ ]:
# ==========================================
# 5. Merging & Renaming
# ==========================================

# cols_map defines which features to extract and their base names
cols_map = {
    'AvgGoals': 'AvgGoals', # GDiff
    'AvgShots': 'AvgShots', # SDiff
    'AvgSoT': 'AvgSoT',     # STDiff
    'AvgCorners': 'AvgCorners', # CDiff
    'AvgFouls': 'AvgFouls', # FDiff
    'WinRate': 'W',         # HomeW / AwayW
    'DrawRate': 'D',        # HomeD / AwayD
    'Off': 'Off',           # HomeOff / AwayOff
    'Def': 'Def',           # HomeDef / AwayDef
    'P': 'P',               # HomeP / AwayP
    'VenueWinRate': 'VenueW', # HomeHW / AwayAW
    'VenueDrawRate': 'VenueD', # HomeHD / AwayAD
    'RestDays': 'RestDays'
}

# Prepare home team features
home_feats = long_df[long_df['IsHome']==1][['Date', 'Team'] + list(cols_map.keys())].copy()
home_feats.columns = ['Date', 'HomeTeam'] + [f'Home{v}' for k,v in cols_map.items()]
# Special fix: HomeVenueW -> HomeHW, HomeVenueD -> HomeHD
home_feats.rename(columns={'HomeVenueW': 'HomeHW', 'HomeVenueD': 'HomeHD'}, inplace=True)

# Prepare away team features
away_feats = long_df[long_df['IsHome']==0][['Date', 'Team'] + list(cols_map.keys())].copy()
away_feats.columns = ['Date', 'AwayTeam'] + [f'Away{v}' for k,v in cols_map.items()]
# Special fix: AwayVenueW -> AwayAW, AwayVenueD -> AwayAD
away_feats.rename(columns={'AwayVenueW': 'AwayAW', 'AwayVenueD': 'AwayAD'}, inplace=True)

# Execute merging
df_final = pd.merge(df, home_feats, on=['Date', 'HomeTeam'], how='left')
df_final = pd.merge(df_final, away_feats, on=['Date', 'AwayTeam'], how='left')

# ==========================================
# 6. Calculate difference features (Diffs Calculation)
# ==========================================

df_final['AvgGDiff'] = df_final['HomeAvgGoals'] - df_final['AwayAvgGoals']
df_final['AvgSDiff'] = df_final['HomeAvgShots'] - df_final['AwayAvgShots']
df_final['AvgSTDiff'] = df_final['HomeAvgSoT'] - df_final['AwayAvgSoT']
df_final['AvgCDiff'] = df_final['HomeAvgCorners'] - df_final['AwayAvgCorners']
df_final['AvgFDiff'] = df_final['HomeAvgFouls'] - df_final['AwayAvgFouls']
df_final['PDiff'] = df_final['HomeP'] - df_final['AwayP']

# Remove NaN (due to rolling calculations in the first few rounds)
df_final = df_final.dropna(subset=['HomeAvgGoals', 'AwayAvgGoals']).reset_index(drop=True)

In [ ]:
# ==========================================
# 7. Final Checks and Output
# ==========================================
print("Generated Columns:", df_final.columns.tolist())

# Check if all desired columns are present
required_checks = [
    'HomeElo', 'EloDiff', 'EloHomeW', # ELO
    'HomeOff', 'AwayDef',             # ODM
    'HomeP', 'PDiff',                 # Points
    'HomeW', 'HomeD', 'HomeHW',       # Stats
    'AvgGDiff', 'AvgSTDiff', 'AvgCDiff' # Form Diffs
]

missing = [c for c in required_checks if c not in df_final.columns]
if not missing:
    print("\nSUCCESS: All requested features are present!")
else:
    print("\nWARNING: Still missing columns:", missing)

# Save
df_final.to_csv('final_feature_engineered_data.csv', index=False)
print("\nFile saved as 'final_feature_engineered_data.csv'")
df_final.head(20)